In [1]:
import time
from typing import Callable

from requests import get, Response


def fetch_content(url: str) -> Response:
    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10.15; rv:155.0) Gecko/20100101 Firefox/155.0'
    }
    response = get(url, headers=headers)

    if response.status_code != 200:
        raise Exception(f"Failed to open url {url}: {response.status_code}, {response.text}")

    return response

def batch_requests(urls: list[str], func: Callable[[Response], None], timeout_ms: float = 0.1):
    for url in urls:
        response = fetch_content(url)
        func(response)
        time.sleep(timeout_ms)

In [2]:
from abc import ABC, abstractclassmethod, abstractmethod
from dataclasses import dataclass

from bs4 import BeautifulSoup


@dataclass
class Page(ABC):
    title: str
    text: str
    full_content: BeautifulSoup
    response: Response
    url: str
    
    def __init__(self, response: Response):
        self.full_content = BeautifulSoup(response.content, "html.parser")
        self.title = self.full_content.find_all("meta", attrs={"property": "og:title"})[0]["content"]
        self.response = response
        self.url = response.url
        self.text = self.get_text_content()
    
    @abstractmethod
    def get_text_content(self) -> str:
        pass
    
    def get_total_size_in_bytes(self) -> int:
        return len(self.response.content)
    
    def get_info_size_in_bytes(self) -> int:
        return len(bytes(self.title + self.text, encoding="utf-8"))
    
    def save_to_file(self, path: str = './page.html') -> None:
        with open(path, 'w') as f:
            f.write(self.full_content.prettify())

@dataclass
class Source(ABC):
    name: str
    url: str
    sample_pages: list[Page]
    page_count: int
    
    @abstractclassmethod
    def count_pages(cls) -> int:
        pass

    @property
    def sample_page(self):
        return self.sample_pages[0]

    def get_average_total_size_in_bytes(self) -> float:
        return round(sum([sample_page.get_total_size_in_bytes() for sample_page in self.sample_pages]) / len(self.sample_pages))

    def get_average_parsed_size_in_bytes(self) -> float:
        return round(sum([sample_page.get_info_size_in_bytes() for sample_page in self.sample_pages]) / len(self.sample_pages))

    def get_approximate_total_size_in_megabytes(self) -> int:
        return round(self.page_count * self.get_average_total_size_in_bytes() / (1024 * 1024))

    def get_approximate_parsed_size_in_megabytes(self) -> int:
        return round(self.page_count * self.get_average_parsed_size_in_bytes() / (1024 * 1024))

In [3]:
@dataclass
class RiaPage(Page):
    def __init__(self, response: Response):
        super().__init__(response)
    
    def get_text_content(self) -> str:
        content_block = self.full_content.find_all("div", attrs={"class": "article__body js-mediator-article mia-analytics"})[0]
        
        div_content_blocks = content_block.find_all("div", attrs={"class": ["article__block"]})
        text_content_blocks = [block for block in div_content_blocks if block["data-type"] == "text" or block["data-type"] == "quote"]
        
        text_blocks = [block.text for block in text_content_blocks]
        return "\n".join(text_blocks)


@dataclass
class RiaSource(Source):
    url = "https://ria.ru"
    
    def __init__(self):
        self.name = "Ria news"
        self.sample_pages = self._get_sample_pages()
        self.page_count = self.count_pages()

    @classmethod
    def _get_sample_pages_urls(cls, sample_size = 100) -> list[str]:
        response = fetch_content(f"{cls.url}/sitemap_article.xml?date_start=20260901&date_end=20260917")
        xml_content = BeautifulSoup(response.content, "xml")
        urls = [item.text for item in xml_content.find_all("loc")][:sample_size]
        return urls
    
    @classmethod
    def _get_sample_pages(cls) -> list[Page]:
        urls = cls._get_sample_pages_urls()
        pages = []
        
        def parser(res: Response):
            nonlocal urls
            
            page = RiaPage(res)
            pages.append(page)
            
        batch_requests(urls, parser)
        return pages
        

    @classmethod
    def _get_article_group_links(cls) -> list[str]:
        response = fetch_content(f"{cls.url}/sitemap_article_index.xml")
        xml_content = BeautifulSoup(response.content, "xml")
        return [item.text for item in xml_content.find_all("loc")]

    @classmethod
    def _get_article_count_for_groups(cls, urls: list[str]) -> int:
        total_count = 0

        def parser(res: Response):
            nonlocal total_count

            xml_content = BeautifulSoup(res.content, "xml")
            found_links = len(xml_content.find_all("url"))
            
            print(f"Found {found_links} for ria source")
            
            total_count += found_links
            

        batch_requests(urls, parser)
        return total_count
    
    @classmethod
    def count_pages(cls) -> int:
        article_group_links = cls._get_article_group_links()
        return cls._get_article_count_for_groups(article_group_links)

## Данные по источнику ria.ru


In [4]:
ria = RiaSource()
ria.sample_page.title, ria.sample_page.text

Found 14076 for ria source
Found 22556 for ria source
Found 22305 for ria source
Found 22593 for ria source
Found 21590 for ria source
Found 20440 for ria source
Found 26207 for ria source
Found 22668 for ria source
Found 22794 for ria source
Found 26222 for ria source
Found 24998 for ria source
Found 23388 for ria source
Found 22651 for ria source
Found 20142 for ria source
Found 21074 for ria source
Found 20250 for ria source
Found 19069 for ria source
Found 18977 for ria source
Found 19260 for ria source
Found 17774 for ria source
Found 15346 for ria source
Found 17597 for ria source
Found 18093 for ria source
Found 19325 for ria source
Found 18607 for ria source
Found 18374 for ria source
Found 19157 for ria source
Found 19214 for ria source
Found 18332 for ria source
Found 16995 for ria source
Found 18886 for ria source
Found 16541 for ria source
Found 16194 for ria source
Found 17709 for ria source
Found 18579 for ria source
Found 18690 for ria source
Found 17499 for ria source
F

('Власти Обнинска объяснили, почему подарили бытовую химию Гайфутдиновой',
 'ЯРОСЛАВЛЬ, 17 сен - РИА Новости. Бытовая химия, врученная мэром Обнинска Стефаном Переваловым в качестве подарка пловчихе Алине Гайфутдиновой, произведена местной компанией, пользуется популярностью у горожан и полезна в быту, рассказали РИА Новости в пресс-службе администрации города. \nГайфутдинова ранее рассказала, что Перевалов подарил ей мочалку и набор бытовых средств: капсулы для стирки, средство от налета и жира, гели для стирки белья, средства для мытья посуды и чистки труб. Спортсменке также подарили умную колонку "Яндекс Станция" и пряник. Гайфутдинова заявила, что счастлива, а подаренное - самые нужные в быту вещи. \n"Мы гордимся нашими спортсменами и стараемся после соревнований встречать их с подарками. Алине вручили полезные в быту подарки: бытовую химию от местного производителя SINTEC Group, которой пользуются многие горожане, умную колонку "Яндекс" и сладкие сувениры", - рассказали РИА Новост

In [5]:
f"Общее число документов = {ria.page_count}"

'Общее число документов = 4453692'

In [38]:
f"Средний размер страницы = {round(ria.get_average_total_size_in_bytes() / 1024)} Kb"

'Средний размер страницы = 179 Kb'

In [39]:
f"Средний размер контента = {round(ria.get_average_parsed_size_in_bytes() / 1024)} Kb"

'Средний размер контента = 2 Kb'

In [6]:
f"Примерный общий размер страниц = {ria.get_approximate_total_size_in_megabytes()} Мб"

'Примерный общий размер страниц = 778118 Мб'

In [7]:
f"Примерный размер контента страниц = {ria.get_approximate_parsed_size_in_megabytes()} Мб"

'Примерный размер контента страниц = 9455 Мб'

Сохранение примера страницы в ria.html

In [8]:
ria.sample_page.save_to_file("ria.html")

Контент с страницы

In [9]:
f"Заголовок документа-примера - \"{ria.sample_page.title}\""

'Заголовок документа-примера - "Власти Обнинска объяснили, почему подарили бытовую химию Гайфутдиновой"'

In [10]:
print(f"Текст документа-примера:\n\"{ria.sample_page.text}\"")

Текст документа-примера:
"ЯРОСЛАВЛЬ, 17 сен - РИА Новости. Бытовая химия, врученная мэром Обнинска Стефаном Переваловым в качестве подарка пловчихе Алине Гайфутдиновой, произведена местной компанией, пользуется популярностью у горожан и полезна в быту, рассказали РИА Новости в пресс-службе администрации города. 
Гайфутдинова ранее рассказала, что Перевалов подарил ей мочалку и набор бытовых средств: капсулы для стирки, средство от налета и жира, гели для стирки белья, средства для мытья посуды и чистки труб. Спортсменке также подарили умную колонку "Яндекс Станция" и пряник. Гайфутдинова заявила, что счастлива, а подаренное - самые нужные в быту вещи. 
"Мы гордимся нашими спортсменами и стараемся после соревнований встречать их с подарками. Алине вручили полезные в быту подарки: бытовую химию от местного производителя SINTEC Group, которой пользуются многие горожане, умную колонку "Яндекс" и сладкие сувениры", - рассказали РИА Новости в администрации. 
Сама девушка в своем Telegram-кан

In [19]:
@dataclass
class IzvestiaPage(Page):
    def __init__(self, response: Response):
        super().__init__(response)

    def get_text_content(self) -> str:
        content_block = self.full_content.find_all("div", attrs={"class": "text-article"})[0]

        text_content_blocks = [block for block in content_block.find_all("p")]
        text_blocks = [block.text for block in text_content_blocks]

        return "\n".join(text_blocks)

@dataclass
class IzvestiaSource(Source):
    url = "https://iz.ru"

    def __init__(self):
        self.name = "Izvestia news"
        self.sample_pages = self._get_sample_pages()
        self.page_count = self.count_pages()

    @classmethod
    def _get_sample_pages_urls(cls, sample_size = 100) -> list[str]:
        response = fetch_content(f"{cls.url}/export/sitemap/1/xml")
        xml_content = BeautifulSoup(response.content, "xml")
        urls = [item.text for item in xml_content.find_all("loc")][:sample_size]
        return urls

    @classmethod
    def _get_sample_pages(cls) -> list[Page]:
        urls = cls._get_sample_pages_urls()
        pages = []

        def parser(res: Response):
            nonlocal urls

            page = IzvestiaPage(res)
            pages.append(page)

        batch_requests(urls, parser)
        return pages

    @classmethod
    def _get_article_group_links(cls) -> list[str]:
        response = fetch_content(f"{cls.url}/sitemap.xml")
        xml_content = BeautifulSoup(response.content, "xml")
        return [item.text for item in xml_content.find_all("loc")]

    @classmethod
    def _get_article_count_for_groups(cls, urls: list[str]) -> int:
        total_count = 0

        def parser(res: Response):
            nonlocal total_count

            xml_content = BeautifulSoup(res.content, "xml")
            found_links = len(xml_content.find_all("url"))

            print(f"Found {found_links} for tass source")

            total_count += found_links
        
        batch_requests(urls, parser)
        return total_count

    @classmethod
    def count_pages(cls) -> int:
        article_group_links = cls._get_article_group_links()
        return cls._get_article_count_for_groups(article_group_links)


## Данные по источнику iz.ru

In [25]:
izvestia_source = IzvestiaSource()
izvestia_source.sample_page.title, izvestia_source.sample_page.text

Found 2999 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 24999 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 25000 for tass source
Found 24998 for tass source
Found 24999 for tass source
Found 24999 for tass source
Found 24999 for tass source
Found 24998 for tass source
Found 24997 for tass source
Found 24995 for tass source
Found 24998 for tass source
Found 24997 for tass source
Found 24996 for tass source
Found 24997 for tass source
Found 24995 for tass source
Found 24998 for tass source
Found 24996 for tass source
Found 25000 for tass source
Found 24997 for tass source
Found 24997 for tass source
Found 24996 for tass source
Found 24998 for tass 

('Минкомсвязи: ограничений на иностранный софт станет больше',
 'В январе в России начнет работать реестр отечественного программного обеспечения. Согласно закону, российские госорганы будут обязаны преимущественно покупать тот софт, который внесен в этот реестр. Если представители госоргана считают, что отечественное программное решение им не подходит, то потребуется обосновать, почему покупают иностранное. По словам главы Минкомсвязи Николая Никифорова, в дальнейшем закон будет расширен на госкорпорации, а когда реестр отечественного ПО будет наполнен, нужно начать отграничивать использование иностранного софта.\nПо словам министра, принятие нормативных автов о реестре отечественного ПО сопровождалось информационной кампанией — о том, что покупать иностранное нужно продолжать. У вводимых ограничений есть ярые противники, прежде всего это сами иностранные поставщики софта, а также ИТ-интеграторы.\nМинистр отметил, что госорганы тратят на закупки ИТ-продукции 80 млрд рублей в год, а го

In [26]:
f"Общее число документов = {izvestia_source.page_count}"

'Общее число документов = 1613574'

In [36]:
f"Средний размер страницы = {round(izvestia_source.get_average_total_size_in_bytes() / 1024)} Kb"

'Средний размер страницы = 168 Kb'

In [37]:
f"Средний размер контента = {round(izvestia_source.get_average_parsed_size_in_bytes() / 1024)} Kb"

'Средний размер контента = 4 Kb'

In [27]:
f"Примерный общий размер страниц = {izvestia_source.get_approximate_total_size_in_megabytes()} Мб"

'Примерный общий размер страниц = 264966 Мб'

In [28]:
f"Примерный размер контента страниц = {izvestia_source.get_approximate_parsed_size_in_megabytes()} Мб"

'Примерный размер контента страниц = 6405 Мб'

Сохранение примера страницы в izvestia.html 

In [29]:
izvestia_source.sample_page.save_to_file("izvestia.html")

Контент с страницы

In [30]:
f"Заголовок документа-примера - \"{izvestia_source.sample_page.title}\""

'Заголовок документа-примера - "Минкомсвязи: ограничений на иностранный софт станет больше"'

In [31]:
print(f"Текст документа-примера:\n\"{izvestia_source.sample_page.text}\"")

Текст документа-примера:
"В январе в России начнет работать реестр отечественного программного обеспечения. Согласно закону, российские госорганы будут обязаны преимущественно покупать тот софт, который внесен в этот реестр. Если представители госоргана считают, что отечественное программное решение им не подходит, то потребуется обосновать, почему покупают иностранное. По словам главы Минкомсвязи Николая Никифорова, в дальнейшем закон будет расширен на госкорпорации, а когда реестр отечественного ПО будет наполнен, нужно начать отграничивать использование иностранного софта.
По словам министра, принятие нормативных автов о реестре отечественного ПО сопровождалось информационной кампанией — о том, что покупать иностранное нужно продолжать. У вводимых ограничений есть ярые противники, прежде всего это сами иностранные поставщики софта, а также ИТ-интеграторы.
Министр отметил, что госорганы тратят на закупки ИТ-продукции 80 млрд рублей в год, а госкомпании — еще больше. По его словам, ко

## Результаты по поисковикам

ria.ru - google.com, yandex.ru

"Глава МИД Украины выдвинул наглое требование после удара ВС России site:ria.ru" - Поисковики выдают одинаковую информацию с нужной страницей наверху

"Глава МИД Украины выдвинул наглое требование после удара ВС России" - Поисковики выдают одинаковую информацию с нужной страницей наверху. Присутствует статья с примерно таким же названием, но более давняя и она находится ниже в google.com в то время как в yandex.ru вообще нет второй статьи

iz.ru - google.com, yandex.ru:

"ФРС США повысила ключевую ставку впервые за три года site:iz.ru" - Поисковики выдают примерно одинаковые результаты, за исключением того, что на 1 строке google.com выдает мобильную версию сайта вместо обычной

"ФРС США повысила ключевую ставку впервые за три года" - google.com не выдает iz.ru в первой десятке результатов, в то время как yandex.ru выдает. google.com выдает не очень популярные источники, что является недостатком поисковика

## Данные о структуре страницы

Метаданные - Присутствуют метаданные, из которых можно получить название и описание статьи

Контент - полезный контент выделен в общую группу и может быть в дальнейшем довольно просто распаршен посредством фильтрации по классу компонента. Однако iz.ru распарсить гораздо проще, чем tass.ru из-за присутствия блоков, не являющихся полезным контентом (в частности краткий пересказ и т.д.)